# Nemotron-4B & Qwen-3.5 Domain Expert Fine-Tuning Pipeline (LoRI-MoE Architecture)
**A comprehensive pipeline for SFT, DPO, and proprietary ranking techniques across Math, Code, and Science domains.**

This notebook utilizes `trl` to train multiple adapters natively on 4B and 0.8B models across various ranks (1, 2, 8, 128, 1024, 3072).

*Note: For ranks > 1024, this script employs strict memory exception handling, as even 32GB VRAM cards may face Out-of-Memory (OOM) errors without DeepSpeed Zero-3 or pipeline parallelism.*


In [ ]:
!pip install -q transformers peft trl datasets bitsandbytes accelerate


In [ ]:
import os
import gc
import torch
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer, DPOConfig, DPOTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [ ]:
DOMAINS = ["math", "code", "science"]
TECHNIQUES = ["SFT", "DPO"]
RANKS = [1, 2, 8, 128, 1024, 3072]

MODELS = {
    "qwen_0.8b": "/home/learner/Desktop/mewtwo/models/Qwen3.5-0.8B", 
    "nemotron_4b": "nvidia/Nemotron-Mini-4B-Instruct"
}


In [ ]:
def get_base_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token
        
    quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        device_map="auto", 
        torch_dtype=torch.bfloat16, 
        quantization_config=quant_config,
        trust_remote_code=True
    )
    return model, tokenizer

def run_experiment(model_key, rank, domain, technique):
    model_name = MODELS[model_key]
    experiment_name = f"{model_key}_{domain}_{technique}_rank{rank}"
    output_dir = f"./outputs/{experiment_name}"
    print(f"\n[{experiment_name}] Starting...")
    
    try:
        model, tokenizer = get_base_model(model_name)
        lora_config = LoraConfig(r=rank, lora_alpha=rank*2, target_modules=["q_proj", "v_proj"], task_type=TaskType.CAUSAL_LM)
        model = get_peft_model(model, lora_config)
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        
        # Proprietary Technique integration placeholder
        # In actual deployment, we apply 'Gate-Conditioned LoRI' constraints here
        
        if technique == "SFT":
            ds = load_dataset("HuggingFaceH4/MATH-500", split="train[:50]") # Mock for speed
            ds = ds.map(lambda x: {"text": x["problem"]})
            
            args = SFTConfig(output_dir=output_dir, max_seq_length=512, per_device_train_batch_size=1, gradient_accumulation_steps=4, dataset_text_field="text")
            trainer = SFTTrainer(model=model, train_dataset=ds, peft_config=lora_config, tokenizer=tokenizer, args=args)
            trainer.train()

        elif technique == "DPO":
            ds = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs[:50]")
            args = DPOConfig(output_dir=output_dir, per_device_train_batch_size=1, gradient_accumulation_steps=4)
            trainer = DPOTrainer(model=model, train_dataset=ds, args=args, tokenizer=tokenizer, peft_config=lora_config)
            trainer.train()
            
        print(f"[{experiment_name}] Done.")
    except torch.OutOfMemoryError:
        print(f"[{experiment_name}] FAILED: OOM.")
    finally:
        if 'model' in locals(): del model
        if 'trainer' in locals(): del trainer
        torch.cuda.empty_cache()
        gc.collect()


In [ ]:
# Run subset of matrix to demonstrate capability
for model in MODELS:
    for domain in ["math", "code", "science"]: 
        for tech in ["SFT"]:
            for rank in [8]: # Use a safe rank for demonstration of merging
                run_experiment(model, rank, domain, tech)

# Merge Domain Adapters
for model_key in MODELS:
    rank = 8
    tech = "SFT"
    print(f"\nMerging adapters for {model_key} {tech} rank {rank}...")
    try:
        model, tokenizer = get_base_model(MODELS[model_key])
        adapters_to_merge = []
        for domain in DOMAINS:
            adapter_path = f"./outputs/{model_key}_{domain}_{tech}_rank{rank}"
            if os.path.exists(adapter_path):
                model.load_adapter(adapter_path, adapter_name=domain)
                adapters_to_merge.append(domain)
                
        if len(adapters_to_merge) > 1:
            weights = [1.0 / len(adapters_to_merge)] * len(adapters_to_merge)
            model.add_weighted_adapter(
                adapters=adapters_to_merge,
                weights=weights,
                adapter_name="merged_multi_domain",
                combination_type="linear"
            )
            model.set_adapter("merged_multi_domain")
            merged_output = f"./outputs/{model_key}_merged_{tech}_rank{rank}"
            model.save_pretrained(merged_output)
            print(f"Successfully saved merged adapter at {merged_output}")
    except Exception as e:
        print(f"Merge skipped/failed: {e}")
    finally:
        if 'model' in locals(): del model
        torch.cuda.empty_cache()
        gc.collect()
